# Laboratorio 10a — Armar la red
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 11 de noviembre de 2026 · Clase 10 (día doble)
**Bloque:** Lab 1 · 10:35–11:40 (65 min), después de la Teoría 1
**Viene de:** Teoría 1 (neurona real → neurona = regresión logística → historia → perceptrón decide y aprende → XOR → apilar → capas ocultas → activaciones → MLP → pérdida → gradiente → regla de la cadena → backpropagation → ciclo de entrenamiento).
**Sigue:** Lab 10b (otro notebook, después de la Teoría 2).

| Parte | Tema | Tiempo sugerido |
|---|---|---|
| **A** | El perceptrón decide y aprende | 12 min |
| **B** | Apilar neuronas, a mano | 8 min |
| **C** | Regla de la cadena y un MLP desde cero | 25 min |
| **D** | Dígitos con PyTorch | 15 min |
| **E** | Comprensión | 5 min |

## Objetivos de aprendizaje

1. Ver que el perceptrón aprende de sus errores, pero no resuelve XOR.
2. Ver que apilar neuronas sí resuelve XOR (a mano y aprendido).
3. Comprobar que la regla de la cadena da los mismos gradientes que las diferencias finitas y que `autograd`.
4. Armar y entrenar un MLP para dígitos con PyTorch: 6.570 parámetros y el ciclo adelante → pérdida → atrás → actualizar.

> **Cómo se trabaja.** Las celdas con `# TODO` se completan; las celdas «Autocomprobación» fallan con un mensaje si algo no está bien implementado. Las preguntas se responden en la celda de texto que sigue a cada una.
>
> **Alcance.** En este laboratorio se entrena solo con **gradiente descendente simple (SGD)** y sin regularización. Los optimizadores (momentum, RMSprop, Adam) y la regularización (L1/L2, Dropout, Batch Normalization, *early stopping*) se ven en la Teoría 2 y se aplican en el Lab 10b.

In [ ]:
import sys, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy scikit-learn matplotlib pandas torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as e:
    raise ImportError("No se encontró PyTorch. En tu computador: activa el entorno virtual e instala con "
                      "`pip install -r requirements.txt` (ver la lámina «Dejen instalando PyTorch ahora»). "
                      "Si no, abre este notebook en Google Colab.") from e
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score

SEED = 42
RAPIDO = False   # True: menos épocas, solo para probar el notebook rápido (valores del libreto = False)
np.random.seed(SEED); torch.manual_seed(SEED)
T0 = time.time()

EPOCAS = {"lab1": 30, "lr_corto": 15}
if RAPIDO:
    EPOCAS = {"lab1": 10, "lr_corto": 5}

AZUL, NARANJA, MARINO, GRIS = "#2e69b2", "#ba6826", "#1b2740", "#8b929c"   # colores de las láminas
plt.rcParams.update({
    "figure.facecolor": "#f2f1ee", "axes.facecolor": "#fffefb", "axes.edgecolor": "#dcdad3",
    "axes.grid": True, "grid.color": "#e3e1da", "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, MARINO, GRIS, "#9aa6b8"]),
    "axes.titlesize": 12, "axes.labelsize": 11, "figure.dpi": 100,
})
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| scikit-learn", sklearn.__version__,
      "| torch", torch.__version__, "| RAPIDO =", RAPIDO)

---
# LAB 1 — Armar la red
---
## Parte A — El perceptrón decide y aprende (12 min)

El **perceptrón** (Rosenblatt, 1958) es una neurona con activación escalón.

* **Decide:** $z = w\cdot x + b$, y $\hat y = 1$ si $z \ge 0$, si no $\hat y = 0$.
* **Aprende,** ejemplo por ejemplo:

$$w \leftarrow w + \eta\,(y - \hat y)\,x, \qquad b \leftarrow b + \eta\,(y - \hat y).$$

Si acierta ($y = \hat y$) no cambia nada; si se equivoca, mueve la recta hacia el ejemplo mal clasificado.

### A1. El ejemplo de la lámina, paso a paso

Datos: $w = (1, -1)$, $b = 0$, $\eta = 1$, $x = (2, 3)$, $y = 1$. En la lámina: $z = -1 \Rightarrow \hat y = 0$ (error) $\Rightarrow w = (3, 2),\ b = 1 \Rightarrow z = 13 \Rightarrow \hat y = 1$.

**Consigna.** Implementa `paso_perceptron(w, b, x, y, eta)`, que aplica la regla con **un** ejemplo y devuelve `z`, `yhat` y los `w`, `b` nuevos. Las celdas de abajo comprueban que reproduces los números de la lámina.

In [ ]:
def escalon(z):
    return np.where(z >= 0, 1, 0)

In [ ]:
def paso_perceptron(w, b, x, y, eta):
    """Un paso de la regla del perceptrón con UN ejemplo. Devuelve (z, yhat, w_nuevo, b_nuevo)."""
    # TODO: calcula z = w·x + b y la predicción yhat con `escalon`;
    #       luego w_nuevo = w + eta*(y - yhat)*x  y  b_nuevo = b + eta*(y - yhat)
    z, yhat, w_nuevo, b_nuevo = None, None, None, None
    return z, yhat, w_nuevo, b_nuevo

In [ ]:
w0, b0, x0 = np.array([1.0, -1.0]), 0.0, np.array([2.0, 3.0])
z1, yh1, w1, b1 = paso_perceptron(w0, b0, x0, 1, 1.0)

# Autocomprobación (los números de la lámina)
assert z1 is not None, "paso_perceptron todavía no está implementada: completa el TODO"
assert (z1, yh1) == (-1.0, 0), f"Con w=(1,-1), b=0 y x=(2,3) se espera z = -1 y yhat = 0; obtuve z = {z1}, yhat = {yh1}"
assert np.allclose(w1, [3, 2]) and b1 == 1, f"Tras el error se espera w = (3, 2) y b = 1; obtuve w = {w1}, b = {b1}"
z2, yh2, _, _ = paso_perceptron(w1, b1, x0, 1, 1.0)
assert (z2, yh2) == (13.0, 1), f"Con los pesos nuevos se espera z = 13 y yhat = 1; obtuve z = {z2}, yhat = {yh2}"
print(f"Antes:   w = (1, -1), b = 0  ->  z = {z1:.0f}  ->  yhat = {yh1}  (y = 1: error)")
print(f"Después: w = ({w1[0]:.0f}, {w1[1]:.0f}), b = {b1:.0f}  ->  z = {z2:.0f}  ->  yhat = {yh2}  (acierta)")

xs = np.linspace(-1, 4, 100)
recta = lambda w, b: -(w[0] * xs + b) / w[1]          # x2 en función de x1 sobre la recta w·x + b = 0
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(xs, recta(w0, b0), ":", color=GRIS, lw=2.5, label="antes: $w=(1,-1)$, $b=0$")
ax.plot(xs, recta(w1, b1), color=AZUL, lw=2.5, label="después: $w=(3,2)$, $b=1$")
ax.scatter(*x0, color=NARANJA, s=140, zorder=3, edgecolor="white", label="$x=(2,3)$, $y=1$")
ax.set_xlim(-1, 4); ax.set_ylim(-3.5, 4.5); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title("Una corrección del perceptrón"); ax.legend(loc="upper left", fontsize=9)
plt.tight_layout(); plt.show()

### A2. Aprender AND, ejemplo por ejemplo

**Consigna.** Completa `entrenar_perceptron(X, y, eta, epocas)`: en cada época recorre los ejemplos en orden, aplica `paso_perceptron` y cuenta los errores (`yhat != y`). Devuelve `(w, b, errores_por_epoca)`. Entrénalo con la compuerta **AND** y verifica que llega a cero errores.

In [ ]:
def graficar_frontera(predecir, X, y, titulo=""):
    """Dibuja la región de decisión de `predecir` (recibe una malla Nx2 y devuelve valores en [0, 1]) y los 4 puntos."""
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
    zz = np.asarray(predecir(np.c_[xx.ravel(), yy.ravel()])).reshape(xx.shape)
    plt.contourf(xx, yy, zz, levels=[-0.01, 0.5, 1.01], colors=["#d9dde6", "#cfe0f3"])
    plt.contour(xx, yy, zz, levels=[0.5], colors=[MARINO], linewidths=2)
    plt.scatter(X[:, 0], X[:, 1], c=np.where(np.ravel(y) == 1, AZUL, MARINO), s=140, edgecolor="white", zorder=3)
    plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.title(titulo); plt.xlim(-0.5, 1.5); plt.ylim(-0.5, 1.5)


X_log = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1])
y_xor = np.array([0, 1, 1, 0])

In [ ]:
def entrenar_perceptron(X, y, eta=0.1, epocas=50, seed=SEED):
    """Regla del perceptrón, ejemplo por ejemplo. Devuelve (w, b, errores_por_epoca)."""
    rng = np.random.default_rng(seed)
    w = rng.normal(0, 0.1, size=X.shape[1])
    b = 0.0
    errores = []
    for _ in range(epocas):
        err = 0
        for xi, yi in zip(X, y):
            # TODO: un paso del perceptrón con (xi, yi) usando `paso_perceptron`;
            #       actualiza w y b, y suma 1 a `err` si la predicción (yhat) fue distinta de yi
            pass
        errores.append(err)
    return w, b, errores

In [ ]:
w_and, b_and, err_and = entrenar_perceptron(X_log, y_and)

# Autocomprobación
assert len(err_and) == 50, "errores_por_epoca debe tener una entrada por época"
assert err_and[0] > 0, "La primera época de AND debería tener errores (parte con pesos casi nulos): ¿completaste el TODO de entrenar_perceptron?"
assert err_and[-1] == 0, "AND es linealmente separable: el perceptrón debería terminar con 0 errores"
primera_sin_error = next(i for i, e in enumerate(err_and, start=1) if e == 0)
print(f"AND: w = {np.round(w_and, 2)}, b = {b_and:.2f} | errores en la época 1: {err_and[0]} | primera época sin errores: {primera_sin_error}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
plt.sca(ax[0]); graficar_frontera(lambda M: escalon(M @ w_and + b_and).astype(float), X_log, y_and, "Frontera $w\\cdot x+b=0$ en AND")
ax[1].plot(range(1, 51), err_and, marker="o", ms=3, color=AZUL); ax[1].set_xlabel("época"); ax[1].set_ylabel("errores"); ax[1].set_title("Errores por época (AND)")
plt.tight_layout(); plt.show()

### A3. El mismo código con XOR

**Consigna.** Entrena con **XOR** (50 épocas) y grafica los errores por época junto a los de AND.

In [ ]:
w_xor, b_xor, err_xor = entrenar_perceptron(X_log, y_xor)
assert err_xor[-1] > 0, "XOR no es linealmente separable: no puede terminar con 0 errores (si ves 0 errores, ¿completaste el TODO de entrenar_perceptron?)"
print("XOR: errores en la última época =", err_xor[-1], "| épocas con 0 errores:", sum(e == 0 for e in err_xor))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
plt.sca(ax[0]); graficar_frontera(lambda M: escalon(M @ w_xor + b_xor).astype(float), X_log, y_xor, "Perceptrón en XOR (última época)")
ax[1].plot(range(1, 51), err_and, label="AND", marker="o", ms=3, color=AZUL)
ax[1].plot(range(1, 51), err_xor, label="XOR", marker="o", ms=3, color=NARANJA)
ax[1].set_xlabel("época"); ax[1].set_ylabel("errores"); ax[1].set_title("Errores por época"); ax[1].legend()
plt.tight_layout(); plt.show()

**Pregunta:** ¿Por qué el perceptrón nunca llega a cero errores en XOR? Relaciónalo con la frase de la lámina: «si una recta puede separar las clases, la regla termina; si no, corrige para siempre».

**Tu respuesta:**

*(escribe aquí)*

## Parte B — Apilar neuronas, a mano (8 min)

Un perceptrón no puede con XOR, pero **varios** sí. Aquí no se entrena nada: los pesos están fijados a mano, como en las láminas «Si una recta no alcanza, usar varias» y «Las capas ocultas resuelven XOR».

### B1. XOR con tres perceptrones de pesos fijos

$$h_1 = \text{escalón}(x_1 + x_2 - 0{,}5)\ (\text{OR}),\qquad h_2 = \text{escalón}(x_1 + x_2 - 1{,}5)\ (\text{AND}),\qquad \hat y = \text{escalón}(h_1 - h_2 - 0{,}5)\ (h_1\ \text{AND NOT}\ h_2).$$

In [ ]:
H1 = escalon(X_log[:, 0] + X_log[:, 1] - 0.5)     # OR
H2 = escalon(X_log[:, 0] + X_log[:, 1] - 1.5)     # AND
Y_B1 = escalon(H1 - H2 - 0.5)                     # h1 AND NOT h2
assert (Y_B1 == y_xor).all(), "La red de tres perceptrones debería calcular XOR"

fig, ax = plt.subplots(figsize=(5.2, 4.5))
x1 = np.linspace(-0.5, 1.5, 50)
ax.fill_between(x1, 0.5 - x1, 1.5 - x1, color=AZUL, alpha=0.15, label="franja: $\\hat y = 1$")
ax.plot(x1, 0.5 - x1, color=AZUL, lw=2, label="$x_1+x_2=0{,}5$ (OR)")
ax.plot(x1, 1.5 - x1, color=NARANJA, lw=2, label="$x_1+x_2=1{,}5$ (AND)")
ax.scatter(X_log[:, 0], X_log[:, 1], c=np.where(y_xor == 1, AZUL, MARINO), s=140, edgecolor="white", zorder=3)
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title("Dos rectas paralelas y una franja"); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

pd.DataFrame({"x1": X_log[:, 0].astype(int), "x2": X_log[:, 1].astype(int), "h1 (OR)": H1, "h2 (AND)": H2,
              "ŷ": Y_B1, "XOR esperado": y_xor})

### B2. La red ReLU de la lámina

$$h_1 = \text{ReLU}(x_1 + x_2),\qquad h_2 = \text{ReLU}(x_1 + x_2 - 1),\qquad \hat y = h_1 - 2h_2.$$

Mira qué pasa con los cuatro puntos en el espacio de las neuronas ocultas $(h_1, h_2)$.

In [ ]:
relu_np = lambda z: np.maximum(0, z)
S = X_log.sum(axis=1)
H = np.c_[relu_np(S), relu_np(S - 1)]              # espacio oculto (h1, h2)
y_B2 = H[:, 0] - 2 * H[:, 1]

assert np.allclose(H, [[0, 0], [1, 0], [1, 0], [2, 1]]), "Los puntos ocultos deberían ser (0,0), (1,0), (1,0) y (2,1)"
assert np.allclose(y_B2, y_xor), "La salida h1 - 2*h2 debería dar XOR"

fig, ax = plt.subplots(figsize=(5.6, 4.5))
h1 = np.linspace(-0.2, 2.4, 50)
ax.plot(h1, (h1 - 0.5) / 2, color=NARANJA, lw=2, label="recta $\\hat y = 0{,}5$")
ax.scatter(H[y_xor == 0, 0], H[y_xor == 0, 1], c=MARINO, s=160, edgecolor="white", zorder=3, label="clase 0")
ax.scatter(H[y_xor == 1, 0], H[y_xor == 1, 1], c=AZUL, s=160, edgecolor="white", zorder=3, label="clase 1")
for (a, b_), txt in [((0, 0), "(0,0)"), ((1, 0), "(0,1) y (1,0)"), ((2, 1), "(1,1)")]:
    ax.annotate(txt, (a, b_), textcoords="offset points", xytext=(8, 10), fontsize=9)
ax.set_xlim(-0.3, 2.5); ax.set_ylim(-0.4, 1.4); ax.set_xlabel("$h_1$"); ax.set_ylabel("$h_2$")
ax.set_title("Espacio oculto: ahora una recta alcanza"); ax.legend(loc="upper left", fontsize=9)
plt.tight_layout(); plt.show()
print("Puntos ocultos:", [tuple(int(v) for v in fila) for fila in H], "| salida ŷ:", y_B2.astype(int))

**Pregunta:** ¿Qué hizo la capa oculta con los cuatro puntos para que ahora una recta alcance?

**Tu respuesta:**

*(escribe aquí)*

## Parte C — Regla de la cadena y un MLP desde cero (25 min)

### C1. La neurona de la lámina, con números

Una neurona sigmoide con pérdida cuadrática: $z = wx + b$, $a = \sigma(z)$, $L = \tfrac12 (a - y)^2$, con $x = 2$, $y = 1$, $w = 0{,}5$, $b = 0$. Se calcula el gradiente **de tres maneras** y deben coincidir:

1. **A mano**, con la regla de la cadena: $\dfrac{\partial L}{\partial w} = \dfrac{\partial L}{\partial a}\cdot\dfrac{\partial a}{\partial z}\cdot\dfrac{\partial z}{\partial w}$.
2. **Diferencias finitas:** $\dfrac{L(w+\varepsilon) - L(w-\varepsilon)}{2\varepsilon}$ con $\varepsilon = 10^{-5}$.
3. **`autograd` de PyTorch:** `requires_grad=True` y `L.backward()`.

In [ ]:
import math
x_c, y_c, w_c, b_c, eta_c = 2.0, 1.0, 0.5, 0.0, 0.5
sig = lambda z: 1 / (1 + math.exp(-z))
L_de = lambda w, b: 0.5 * (sig(w * x_c + b) - y_c) ** 2

# 1. Hacia adelante
z_c = w_c * x_c + b_c
a_c = sig(z_c)
L_c = 0.5 * (a_c - y_c) ** 2

# 2. Hacia atrás, a mano (producto de derivadas locales)
dL_da = a_c - y_c                  # ∂L/∂a
da_dz = a_c * (1 - a_c)            # ∂a/∂z
dz_dw = x_c                        # ∂z/∂w
dL_dw = dL_da * da_dz * dz_dw
dL_db = dL_da * da_dz * 1.0        # ∂z/∂b = 1

# 3. Verificación 1: diferencias finitas
eps = 1e-5
fd_w = (L_de(w_c + eps, b_c) - L_de(w_c - eps, b_c)) / (2 * eps)
fd_b = (L_de(w_c, b_c + eps) - L_de(w_c, b_c - eps)) / (2 * eps)

# 4. Verificación 2: autograd
wt = torch.tensor(w_c, dtype=torch.float64, requires_grad=True)
bt = torch.tensor(b_c, dtype=torch.float64, requires_grad=True)
Lt = 0.5 * (torch.sigmoid(wt * x_c + bt) - y_c) ** 2
Lt.backward()

# Autocomprobación
assert abs(z_c - 1.0) < 1e-12 and abs(a_c - 0.731) < 1e-3 and abs(L_c - 0.0362) < 1e-4, "Pasada hacia adelante: z = 1, a ≈ 0,731, L ≈ 0,0362"
assert abs(dL_da + 0.269) < 1e-3 and abs(da_dz - 0.197) < 1e-3, "Derivadas locales: ∂L/∂a ≈ -0,269 y ∂a/∂z ≈ 0,197"
for nombre, a_mano, fd, ag in [("w", dL_dw, fd_w, wt.grad.item()), ("b", dL_db, fd_b, bt.grad.item())]:
    assert abs(a_mano - fd) < 1e-4 and abs(a_mano - ag) < 1e-4, f"Los tres gradientes de {nombre} deberían coincidir"

print(f"Adelante: z = {z_c:.3f}  ->  a = {a_c:.3f}  ->  L = {L_c:.4f}")
print(f"Atrás:    ∂L/∂a = {dL_da:.3f} | ∂a/∂z = {da_dz:.3f} | ∂z/∂w = {dz_dw:.0f}")
pd.DataFrame({"a mano": [dL_dw, dL_db], "diferencias finitas": [fd_w, fd_b], "autograd": [wt.grad.item(), bt.grad.item()]},
             index=["∂L/∂w", "∂L/∂b"]).round(6)

In [ ]:
# 5. Un paso de gradiente descendente con η = 0,5
w_nuevo = w_c - eta_c * dL_dw
b_nuevo = b_c - eta_c * dL_db
L_nueva = L_de(w_nuevo, b_nuevo)
assert round(w_nuevo, 3) == 0.553 and round(b_nuevo, 3) == 0.026, "Se espera w = 0,553 y b = 0,026"
print(f"w: {w_c} -> {w_nuevo:.3f} | b: {b_c} -> {b_nuevo:.3f} | pérdida: {L_c:.4f} -> {L_nueva:.4f}")

> **Esto es lo que hace `loss.backward()` en cada peso de una red:** multiplicar derivadas locales a lo largo del camino que lleva desde el peso hasta la pérdida. Con millones de pesos se hace igual, pero reutilizando el error de la capa siguiente.

### C2. Un MLP 2-4-1 en NumPy para XOR

Red **2-4-1**: capa oculta de 4 neuronas con **tanh**, salida con **sigmoide**, pérdida **entropía cruzada binaria** (media sobre los $n$ ejemplos), gradiente descendente de lote completo con $\eta = 0{,}5$.

**Hacia adelante**

$$z_1 = XW_1 + b_1,\quad a_1 = \tanh(z_1),\quad z_2 = a_1W_2 + b_2,\quad \hat y = a_2 = \sigma(z_2).$$

**Hacia atrás** (regla de la cadena, con $n$ = número de ejemplos). Para sigmoide + entropía cruzada la derivada se simplifica:

$$\delta_2 = \frac{\hat y - y}{n},\quad \frac{\partial L}{\partial W_2} = a_1^\top\delta_2,\quad \frac{\partial L}{\partial b_2} = \sum \delta_2,\quad \delta_1 = (\delta_2 W_2^\top)\odot(1-a_1^2),\quad \frac{\partial L}{\partial W_1} = X^\top\delta_1,\quad \frac{\partial L}{\partial b_1} = \sum\delta_1.$$

**Consigna.**
1. Completa `backward` (`forward`, `perdida` y `paso` ya están escritas).
2. **Verifica tus gradientes** con diferencias finitas: el error relativo máximo debe ser menor que $10^{-6}$.
3. Entrena (`entrenar_xor`), grafica la curva de pérdida y la frontera de decisión aprendida.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def init_params(n_in=2, n_oc=4, n_out=1, seed=SEED):
    rng = np.random.default_rng(seed)
    return {"W1": rng.normal(0, 1, (n_in, n_oc)), "b1": np.zeros((1, n_oc)),
            "W2": rng.normal(0, 1, (n_oc, n_out)), "b2": np.zeros((1, n_out))}


y_col = y_xor.reshape(-1, 1).astype(float)   # las etiquetas como columna (n, 1)


def forward(p, X):
    """Paso hacia adelante. Devuelve (yhat, cache) con las activaciones que necesita backward."""
    z1 = X @ p["W1"] + p["b1"]
    a1 = np.tanh(z1)
    z2 = a1 @ p["W2"] + p["b2"]
    a2 = sigmoide(z2)
    return a2, {"X": X, "z1": z1, "a1": a1, "z2": z2, "a2": a2}


def perdida(y, yhat, eps=1e-12):
    """Entropía cruzada binaria media."""
    yhat = np.clip(yhat, eps, 1 - eps)
    return -np.mean(y * np.log(yhat) + (1 - y) * np.log(1 - yhat))


def paso(p, grads, eta):
    """Un paso de gradiente descendente (modifica `p` en el lugar)."""
    for k in p:
        p[k] -= eta * grads[k]

In [ ]:
def backward(p, cache, y):
    """Gradientes de la pérdida respecto de W1, b1, W2, b2 (mismo formato que `p`)."""
    n = y.shape[0]
    # TODO: regla de la cadena, de la salida hacia la entrada (fórmulas de arriba):
    #   dz2 = (a2 - y) / n          -> dW2 = a1.T @ dz2 ,  db2 = suma de dz2 por columnas (keepdims=True)
    #   dz1 = (dz2 @ W2.T) * (1 - a1**2)  -> dW1 = X.T @ dz1 ,  db1 = suma de dz1 por columnas (keepdims=True)
    return None   # devuelve {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

In [ ]:
def verificar_gradientes(p, X, y, eps=1e-5):
    """Compara backward() con diferencias finitas centradas. Devuelve el error relativo máximo."""
    yhat, cache = forward(p, X)
    grads = backward(p, cache, y)
    assert grads is not None, "backward() todavía no devuelve los gradientes (revisa el TODO)."
    peor = 0.0
    for k in p:
        for idx in np.ndindex(*p[k].shape):
            original = p[k][idx]
            p[k][idx] = original + eps; lp = perdida(y, forward(p, X)[0])
            p[k][idx] = original - eps; lm = perdida(y, forward(p, X)[0])
            p[k][idx] = original
            g_num = (lp - lm) / (2 * eps)
            g_an = grads[k][idx]
            peor = max(peor, abs(g_num - g_an) / max(abs(g_num) + abs(g_an), 1e-12))
    return peor


# Autocomprobación de gradientes (con pesos aleatorios, antes de entrenar)
p_test = init_params()
yhat0, _ = forward(p_test, X_log)
assert yhat0.shape == (4, 1), "forward debe devolver un vector columna (n, 1)"
err_rel = verificar_gradientes(p_test, X_log, y_col)
print(f"Error relativo máximo entre backward() y diferencias finitas: {err_rel:.2e}")
assert err_rel < 1e-6, "Los gradientes no coinciden con las diferencias finitas: revisa backward()"
print("Gradientes correctos.")

In [ ]:
def entrenar_xor(eta=0.5, iteraciones=2000, seed=SEED):
    """Entrena la red 2-4-1 en XOR. Devuelve (parametros, historial_de_perdida)."""
    p = init_params(seed=seed)
    hist = []
    for _ in range(iteraciones):
        yhat, cache = forward(p, X_log)
        hist.append(perdida(y_col, yhat))
        grads = backward(p, cache, y_col)
        paso(p, grads, eta)
    return p, hist


p_xor, hist_xor = entrenar_xor()
pred_xor = (forward(p_xor, X_log)[0] > 0.5).astype(int)

# Autocomprobación
assert hist_xor[-1] < 0.05, f"La pérdida final ({hist_xor[-1]:.3f}) debería ser < 0.05"
assert (pred_xor.ravel() == y_xor).all(), "La red debería clasificar bien los 4 puntos de XOR"
print(f"Pérdida final: {hist_xor[-1]:.5f} | predicciones: {pred_xor.ravel()} | objetivo: {y_xor}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist_xor, color=AZUL); ax[0].set_yscale("log"); ax[0].set_xlabel("iteración"); ax[0].set_ylabel("entropía cruzada"); ax[0].set_title("Curva de pérdida (XOR)")
plt.sca(ax[1]); graficar_frontera(lambda M: forward(p_xor, M)[0], X_log, y_xor, "Frontera aprendida por el MLP 2-4-1")
plt.tight_layout(); plt.show()

Ahora la inicialización. Entrena la misma red con distintas semillas y mira dónde termina la pérdida.

In [ ]:
filas = []
for s in [0, 1, 2, 3, 42]:
    p_s, h_s = entrenar_xor(seed=s)
    pred_s = (forward(p_s, X_log)[0] > 0.5).astype(int).ravel()
    filas.append({"semilla": s, "pérdida final": h_s[-1], "aciertos (de 4)": int((pred_s == y_xor).sum())})
pd.DataFrame(filas).round(4)

**Pregunta:** Compara esta frontera con la de la Parte B: ¿en qué se parecen y en qué no? Si cambias la semilla, ¿qué pasa con la pérdida y por qué importa la inicialización?

**Tu respuesta:**

*(escribe aquí)*

## Parte D — Dígitos con PyTorch (15 min)

Conjunto `load_digits` (1.797 imágenes de 8×8 píxeles, 10 clases), incluido en scikit-learn. Se escala a $[0,1]$ y se separa en entrenamiento/prueba (75 % / 25 %, estratificado).

**Consigna.**
1. Define la red `64 → 64 → 32 → 10` con ReLU (`nn.Sequential`) y comprueba que tiene **6.570 parámetros**.
2. Escribe el ciclo de entrenamiento de una época (`entrenar_una_epoca`), igual al de la lámina: *adelante → pérdida → atrás → actualizar*, con mini-batches de 64 (`DataLoader`).
3. Entrena con `nn.CrossEntropyLoss` y `SGD(lr=0.1)`, registra pérdida y exactitud por época, y reporta **matriz de confusión** y **F1 macro** (clase 9).

In [ ]:
digits = load_digits()
X_all, y_all = digits.data / 16.0, digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.25, stratify=y_all, random_state=SEED)

def a_tensores(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

Xtr_t, ytr_t = a_tensores(X_tr, y_tr)
Xte_t, yte_t = a_tensores(X_te, y_te)
print("entrenamiento:", Xtr_t.shape, "| prueba:", Xte_t.shape)

fig, ax = plt.subplots(1, 8, figsize=(12, 2))
for a, img, lab in zip(ax, digits.images[:8], digits.target[:8]):
    a.imshow(img, cmap="Greys"); a.set_title(lab); a.axis("off")
plt.show()

In [ ]:
def crear_red_base():
    """Red 64 -> 64 -> 32 -> 10 con ReLU."""
    return nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))


torch.manual_seed(SEED)
modelo = crear_red_base()
n_params = sum(p.numel() for p in modelo.parameters())
assert n_params == 6570, "La red 64-64-32-10 debe tener 64*64+64 + 64*32+32 + 32*10+10 = 6570 parámetros"
print(modelo)
print(f"Parámetros: ({64}·{64}+{64}) + ({64}·{32}+{32}) + ({32}·{10}+{10}) = {4160} + {2080} + {330} = {n_params}")

In [ ]:
def evaluar(modelo, X, y):
    """Pérdida, exactitud y predicciones del modelo en modo evaluación (sin gradientes)."""
    modelo.eval()
    with torch.no_grad():
        logits = modelo(X)
        perd = nn.functional.cross_entropy(logits, y).item()
        pred = logits.argmax(dim=1)
    return {"loss": perd, "acc": (pred == y).float().mean().item(), "pred": pred.numpy()}

In [ ]:
def entrenar_una_epoca(modelo, optimizador, loader, criterio):
    """Una pasada por el DataLoader (mini-batches). Devuelve la pérdida media de la época."""
    modelo.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        # TODO: las 4 líneas del ciclo de entrenamiento de la lámina
        #   1 · 2   vaciar los gradientes (optimizador.zero_grad()) y calcular la pérdida: loss = criterio(modelo(xb), yb)
        #   3       paso hacia atrás: loss.backward()
        #   4       actualizar los pesos: optimizador.step()
        loss = None
        total += loss.item() * len(xb)
        n += len(xb)
    return total / n

In [ ]:
# Autocomprobación: una época sobre un lote pequeño debe cambiar los pesos y devolver una pérdida
_m = nn.Linear(64, 10)
_w0 = _m.weight.detach().clone()
_loader = DataLoader(TensorDataset(Xtr_t[:128], ytr_t[:128]), batch_size=64)
try:
    _perd = entrenar_una_epoca(_m, torch.optim.SGD(_m.parameters(), lr=0.1), _loader, nn.CrossEntropyLoss())
except AttributeError:
    raise AssertionError("entrenar_una_epoca todavía no está completa: revisa el TODO (¿asignaste `loss`?)") from None
assert not torch.equal(_w0, _m.weight.detach()), "Los pesos no cambiaron: ¿llamaste a loss.backward() y a optimizador.step()?"
print(f"Una época de prueba sobre 128 ejemplos: pérdida media = {_perd:.3f} (≈ ln 10 = 2,303 al inicio) | los pesos cambiaron.")

In [ ]:
def entrenar(modelo, optimizador, X_tr, y_tr, X_ev, y_ev, epocas=30, batch_size=64, seed=SEED):
    """Entrena `epocas` épocas con mini-batches. Devuelve el historial por época:
    loss_tr, acc_tr (entrenamiento, en modo evaluación) y loss_ev, acc_ev (conjunto de evaluación)."""
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, generator=g, drop_last=True)
    criterio = nn.CrossEntropyLoss()
    hist = {"loss_tr": [], "acc_tr": [], "loss_ev": [], "acc_ev": []}
    for _ in range(epocas):
        entrenar_una_epoca(modelo, optimizador, loader, criterio)
        r_tr, r_ev = evaluar(modelo, X_tr, y_tr), evaluar(modelo, X_ev, y_ev)
        hist["loss_tr"].append(r_tr["loss"]); hist["acc_tr"].append(r_tr["acc"])
        hist["loss_ev"].append(r_ev["loss"]); hist["acc_ev"].append(r_ev["acc"])
    return hist


torch.manual_seed(SEED)
modelo = crear_red_base()
opt = torch.optim.SGD(modelo.parameters(), lr=0.1)
hist1 = entrenar(modelo, opt, Xtr_t, ytr_t, Xte_t, yte_t, epocas=EPOCAS["lab1"])

# Autocomprobación
assert hist1["loss_tr"][-1] < hist1["loss_tr"][0], "La pérdida de entrenamiento debería bajar"
assert hist1["acc_ev"][-1] > (0.9 if not RAPIDO else 0.8), "Con esta red y SGD lr=0.1 se espera más de 90 % de exactitud en prueba"
print(f"Exactitud de prueba: {hist1['acc_ev'][-1]:.3f} | pérdida de entrenamiento: {hist1['loss_tr'][0]:.3f} -> {hist1['loss_tr'][-1]:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist1["loss_tr"], label="entrenamiento"); ax[0].plot(hist1["loss_ev"], label="prueba"); ax[0].set_title("Pérdida"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(hist1["acc_tr"], label="entrenamiento"); ax[1].plot(hist1["acc_ev"], label="prueba"); ax[1].set_title("Exactitud"); ax[1].set_xlabel("época"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
pred_te = evaluar(modelo, Xte_t, yte_t)["pred"]
f1m = f1_score(y_te, pred_te, average="macro")
print(f"F1 macro en prueba: {f1m:.3f}")
ConfusionMatrixDisplay(confusion_matrix(y_te, pred_te), display_labels=range(10)).plot(cmap="Blues", colorbar=False)
plt.title("Matriz de confusión (prueba)"); plt.grid(False); plt.show()

# El par de dígitos más confundido
cm = confusion_matrix(y_te, pred_te); np.fill_diagonal(cm, 0)
i, j = np.unravel_index(cm.argmax(), cm.shape)
print(f"Confusión más frecuente: real {i} -> predicho {j} ({cm[i, j]} casos)")

## Parte E — Comprensión (5 min)

Con la función `entrenar` ya lista, responde experimentando.

1. **Tasa de aprendizaje.** Entrena 15 épocas con `SGD` y `lr ∈ {1e-4, 0.1, 5}` (misma semilla) y grafica las tres curvas de pérdida de entrenamiento.
2. **Sin ReLU.** Entrena la misma red sin las activaciones (solo `Linear → Linear → Linear`) y compara la exactitud de prueba con la red con ReLU.

In [ ]:
curvas_lr = {}
for lr in [1e-4, 0.1, 5.0]:
    torch.manual_seed(SEED)
    m = crear_red_base()
    curvas_lr[lr] = entrenar(m, torch.optim.SGD(m.parameters(), lr=lr), Xtr_t, ytr_t, Xte_t, yte_t, epocas=EPOCAS["lr_corto"])

for lr, color in zip(curvas_lr, [GRIS, AZUL, NARANJA]):
    plt.plot(curvas_lr[lr]["loss_tr"], label=f"lr = {lr:g}", color=color, lw=2)
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida de entrenamiento"); plt.title("Efecto de la tasa de aprendizaje (SGD)"); plt.legend(); plt.show()
for lr, h in curvas_lr.items():
    print(f"lr = {lr:g}: pérdida final {h['loss_tr'][-1]:.4f} | exactitud de prueba {h['acc_ev'][-1]:.3f}")

**Pregunta:** ¿Qué pasa con la pérdida si `lr = 5`? ¿Y si `lr = 1e-4`? Relaciónalo con la lámina «Gradiente descendente» (η pequeña: lento; η grande: diverge).

**Tu respuesta:**

*(escribe aquí)*

In [ ]:
torch.manual_seed(SEED)
sin_relu = nn.Sequential(nn.Linear(64, 64), nn.Linear(64, 32), nn.Linear(32, 10))
h_lin = entrenar(sin_relu, torch.optim.SGD(sin_relu.parameters(), lr=0.1), Xtr_t, ytr_t, Xte_t, yte_t, epocas=EPOCAS["lab1"])
print(f"Con ReLU:  exactitud de prueba = {hist1['acc_ev'][-1]:.3f}")
print(f"Sin ReLU:  exactitud de prueba = {h_lin['acc_ev'][-1]:.3f}")

# Tres capas lineales apiladas equivalen a UNA sola capa lineal: W = W3 W2 W1
with torch.no_grad():
    L1, L2, L3 = sin_relu[0], sin_relu[1], sin_relu[2]
    W_eq = L3.weight @ L2.weight @ L1.weight
    b_eq = L3.weight @ (L2.weight @ L1.bias + L2.bias) + L3.bias
    una_capa = Xte_t @ W_eq.T + b_eq
    equivalente = torch.allclose(sin_relu(Xte_t), una_capa, atol=1e-4)
print("¿Salida idéntica a la de una sola capa lineal?", equivalente)
assert equivalente, "Sin activaciones, la composición de capas lineales debe ser una sola capa lineal"

**Pregunta:** ¿Qué ocurre si se eliminan las ReLU? Compara las exactitudes y usa la última celda (la equivalencia con una sola capa lineal). ¿Contradice lo que viste con XOR?

**Tu respuesta:**

*(escribe aquí)*

---
## Entregable

El notebook **`Lab10a_Red_estudiante.ipynb` ejecutado**, con los `TODO` completados y las preguntas respondidas. El Lab 2 de hoy (optimizadores y regularización) se hace en otro notebook, `Lab10b`; al final del día se entregan los dos.

In [ ]:
print(f"Tiempo total de ejecución: {time.time() - T0:.0f} s")